In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS landing;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

CREATE VOLUME IF NOT EXISTS workspace.landing.dados;

In [0]:
SUPABASE_HOST = "aws-1-us-east-1.pooler.supabase.com"
SUPABASE_PORT = "6543"
SUPABASE_DATABASE = "postgres"
SUPABASE_USER = "postgres.mcgfudhndogqzhvhsvji"
SUPABASE_PASSWORD = "mtP6JWmroPKE1sK8"

jdbc_url = (
    f"jdbc:postgresql://{SUPABASE_HOST}:"
    f"{SUPABASE_PORT}/{SUPABASE_DATABASE}"
    "?sslmode=require"
)

def read_supabase_table(table_name):
    return (
        spark.read
            .format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", f"public.{table_name}")
            .option("user", SUPABASE_USER)
            .option("password", SUPABASE_PASSWORD)
            .option("driver", "org.postgresql.Driver")
            .load()
    )

In [0]:
df_test = read_supabase_table("artistas")
df_test.show()

In [0]:
df_test = read_supabase_table("artistas")
df_test.show()

In [0]:
from datetime import datetime

TABLES = [
    'artistas',
    'albuns',
    'musicas',
    'usuarios',
    'reproducoes'
]

LANDING_SCHEMA = 'landing'
LANDING_BASE_PATH = "/Volumes/workspace/landing/dados"
EXTRACTION_DATE = datetime.now().strftime('%Y-%m-%d')

print('Tabelas para extrair:')
for table in TABLES:
    print(f'- {table}')

print(f'Landing path: {LANDING_BASE_PATH}')
print(f'Data da extração: {EXTRACTION_DATE}')

In [0]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {LANDING_SCHEMA}')
spark.sql(f'USE {LANDING_SCHEMA}')

print(f'Schema `{LANDING_SCHEMA}` pronto.')

In [0]:
def read_supabase_table(table_name: str):
    return (
        spark.read
            .format('jdbc')
            .option('url', jdbc_url)
            .option('dbtable', f'public.{table_name}')
            .option('user', SUPABASE_USER)
            .option('password', SUPABASE_PASSWORD)
            .option('driver', 'org.postgresql.Driver')
            .load()
    )

In [0]:
from pyspark.sql.functions import current_timestamp, lit

results = []

for table in TABLES:
    print(f'Extraindo tabela: {table}')

    df = read_supabase_table(table)

    df_landing = (
        df
        .withColumn('_source_table', lit(table))
        .withColumn('_extracted_at', current_timestamp())
    )

    output_path = f'{LANDING_BASE_PATH}/{table}'

    (
       df_landing.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"{LANDING_BASE_PATH}/{table}")
    )

    count = df_landing.count()
    columns_count = len(df_landing.columns)

    results.append({
        'table': table,
        'rows': count,
        'columns': columns_count,
        'path': output_path
    })

    print(f'  OK: {count} registros | {columns_count} colunas | {output_path}')

print('Extração finalizada.')


In [0]:

%sql
SHOW SCHEMAS IN workspace;

In [0]:
for item in results:
    table = item['table']
    path = item['path']

    df = (
        spark.read
            .option('header', 'true')
            .option('inferSchema', 'true')
            .csv(path)
    )

    (
        df.write
            .mode('overwrite')
            .saveAsTable(f'{LANDING_SCHEMA}.{table}')
    )

    print(f'Tabela criada: {LANDING_SCHEMA}.{table}')